# S11 · Volumen promedio de aeronaves en vuelo

**Leonardo Jesus Huaman Arhuata** · Equipo 04 · Big Data · Ciclo 9  
Abel Ángel Sullón Macalupú · Universidad Peruana Unión · Juliaca · 2026-2

El analista verifica qué está llegando desde OpenSky y compara el pronóstico de **aeronaves** con el valor real de la ventana objetivo. [Abrir tablero BI/ML](http://localhost:43000/d/aerostreamperu-s11). Sigue el arranque de S11 en la [guía](README.md).

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import json
import pandas as pd
from consultar_prometheus import consultar, dos_valores_distintos

INTEGRANTE = "leonardo"
LIMA = ZoneInfo("America/Lima")
CONTRATO = Path("../s05-evaluacion-unidad-1/artifacts") / INTEGRANTE / "contrato_modelo_historico.json"
contrato = json.loads(CONTRATO.read_text(encoding="utf-8"))

def serie(nombre, grupo=INTEGRANTE):
    return f'aerostream_bi_{nombre}{{job="aerostream_bi_{grupo}"}}'

def valor(nombre, grupo=INTEGRANTE):
    resultados = consultar(query=serie(nombre, grupo))
    return float(resultados[0]["value"][1]) if resultados else None

def hora_lima(segundos):
    return datetime.fromtimestamp(segundos, LIMA).isoformat(timespec="seconds") if segundos is not None else None

## A · Foto batch del Parquet de S9

Los KPIs corresponden al corte que Spark procesó con `--publicar-kpis`. Grafana consulta estos mismos valores de Prometheus.

In [ ]:
batch = "batch"
foto = {
    "Capturas procesadas": valor("batch_capturas", batch),
    "Proporción de capturas válidas": valor("batch_capturas_validas_ratio", batch),
    "Latencia media de telemetría (s)": valor("batch_latencia_segundos", batch),
    "Última captura incluida (Lima)": hora_lima(valor("batch_ultima_captura_segundos", batch)),
    "Cálculo batch (Lima)": hora_lima(valor("batch_calculado_segundos", batch)),
}
assert foto["Capturas procesadas"] is not None, "Ejecuta la publicación batch de S11 y espera el scrape de Prometheus."
pd.Series(foto, name="Evidencia S9")

## B · Indicador y pronóstico en vivo

La variable objetivo es `volumen_promedio_t1`. En S10 se carga el modelo de **regresión lineal base** guardado en S5. El método preferido por RMSE de validación es **persistencia**.

S11 consume las ventanas de S8 y las predicciones de S10 desde Kafka. Cada micro-lote con registros actualiza Pushgateway. El error usa el valor real de la misma ventana objetivo cuando reúne al menos cuatro capturas y el pronóstico tiene una emisión anterior a su cierre.

In [ ]:
print("Variable objetivo:", contrato["variable_objetivo"])
print("Predictoras:", ", ".join(v["nombre"] for v in contrato["variables_predictoras"]))
print("Modelo MLlib:", contrato["estimador_mllib"])
print("Preferido en validación:", contrato["preferido_en_validacion"])

In [ ]:
ultima = {
    "Captura OpenSky (Lima)": hora_lima(valor("captura_inicio_segundos", "captura")),
    "Lectura de la captura (aeronaves)": valor("captura_aeronaves_en_vuelo", "captura"),
    "Inicio de la última observación (Lima)": hora_lima(valor("observado_inicio_segundos")),
    "Observado (aeronaves)": valor("observado"),
    "Capturas de la observación": valor("observado_capturas"),
    "Inicio de la ventana objetivo (Lima)": hora_lima(valor("pronostico_objetivo_segundos")),
    "Emisión del pronóstico (Lima)": hora_lima(valor("pronostico_emitido_segundos")),
    "MLlib (aeronaves)": valor("pronostico_mllib"),
    "Persistencia (aeronaves)": valor("pronostico_persistencia"),
    "Preferido (aeronaves)": valor("pronostico_preferido"),
}
pd.Series(ultima, name=INTEGRANTE)

In [ ]:
inicio_evaluado = valor("evaluado_objetivo_segundos")
if inicio_evaluado is None:
    print("Todavía no llegó una ventana objetivo apta para medir el error.")
else:
    print("Objetivo evaluado (Lima):", hora_lima(inicio_evaluado))
    print("Valor real:", valor("evaluado_real"), "aeronaves")
    display(pd.DataFrame([
        {"Pronóstico": "MLlib", "Valor": valor("evaluado_mllib"), "Error absoluto": valor("error_absoluto_mllib")},
        {"Pronóstico": "Persistencia", "Valor": valor("evaluado_persistencia"), "Error absoluto": valor("error_absoluto_persistencia")},
    ]))

## C · Comprobar el cambio con PromQL

El cuaderno busca en las últimas tres horas dos instantes con pronósticos distintos. Después ejecuta la misma consulta instantánea de PromQL en esos dos horarios. Puedes repetir la consulta en Prometheus usando los horarios que imprime la tabla.

In [ ]:
expresion = serie("pronostico_mllib")
print(expresion)
muestras = dos_valores_distintos(expresion)
if len(muestras) < 2:
    print("Aún no aparecen dos pronósticos distintos. Repite la consulta cuando cierre otra ventana apta.")
else:
    assert float(muestras[0][1]) != float(muestras[1][1])
display(pd.DataFrame([
    {"Consulta (Lima)": hora_lima(instante), "Pronóstico (aeronaves)": float(resultado)}
    for instante, resultado in muestras
]))

## D · Tablero y hallazgo

En [Grafana](http://localhost:43000/d/aerostreamperu-s11), abre la dimensión de Leonardo y conserva una captura con los KPIs batch y la serie en vivo. Los KPIs cambian al volver a ejecutar el batch. Los pronósticos cambian cuando el flujo cierra otra ventana con suficiente cobertura.

El watermark retrasa el cierre de la ventana de origen. Verifica la hora de emisión frente al inicio y fin de la ventana objetivo para interpretar si el pronóstico se publicó a tiempo. Pushgateway conserva el último valor que recibió, aun si se detiene Spark. La hora de la última captura y el tablero de S9 ayudan a detectar ese caso.